# 04. Statistical Analysis

Statistical analysis is used to determine whether the observed differences and relationships in delivery performance are statistically significant.

### Business Objective

Validate whether key operational factors have a statistically significant relationship with delivery time and late deliveries.

### Key Analysis Areas

- Traffic and delivery time
- Weather and delivery time
- Multiple deliveries and delivery time
- Vehicle condition and delivery time
- Delivery partner ratings and delivery time
- Statistical significance testing

In [3]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("../data/cleaned/food_delivery_cleaned.csv")

df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,1900-01-01 11:30:00,1900-01-01 11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,24
1,0xb379,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,1900-01-01 19:45:00,1900-01-01 19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,33
2,0x5d6d,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,1900-01-01 08:30:00,1900-01-01 08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,26
3,0x7a6a,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,1900-01-01 18:00:00,1900-01-01 18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,21
4,0x70a2,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,1900-01-01 13:30:00,1900-01-01 13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,30


## 1. Traffic vs Delivery Time

Test whether average delivery time differs significantly across different traffic conditions.

### Key Business Questions

- Does traffic condition significantly affect delivery time?
- Are the differences in average delivery time statistically significant?
- Should traffic be considered an important operational factor?

In [4]:
traffic_groups = [
    group["Time_taken(min)"].dropna()
    for _, group in df.groupby("Road_traffic_density")
]

f_stat, p_value = stats.f_oneway(*traffic_groups)

print("F-statistic:", round(f_stat, 3))
print("P-value:", p_value)

F-statistic: 2550.296
P-value: 0.0


## 2. Weather vs Delivery Time

Test whether average delivery time differs significantly across different weather conditions.

### Key Business Questions

- Does weather condition significantly affect delivery time?
- Are the differences between weather groups statistically significant?
- Should weather conditions be considered when estimating delivery times?

In [5]:
weather_groups = [
    group["Time_taken(min)"].dropna()
    for _, group in df.groupby("Weatherconditions")
]

f_stat, p_value = stats.f_oneway(*weather_groups)

print("F-statistic:", round(f_stat, 3))
print("P-value:", p_value)

F-statistic: 608.22
P-value: 0.0


## 3. Multiple Deliveries vs Delivery Time

Test whether the number of deliveries assigned to a delivery partner is significantly associated with delivery time.

### Key Business Questions

- Does the number of multiple deliveries significantly affect delivery time?
- Are delivery times significantly different across multiple-delivery groups?
- Should multiple deliveries be considered when assigning orders?

In [6]:
multiple_delivery_groups = [
    group["Time_taken(min)"].dropna()
    for _, group in df.groupby("multiple_deliveries")
]

f_stat, p_value = stats.f_oneway(*multiple_delivery_groups)

print("F-statistic:", round(f_stat, 3))
print("P-value:", p_value)

F-statistic: 2623.49
P-value: 0.0


## 4. Vehicle Condition vs Delivery Time

Test whether average delivery time differs significantly across vehicle-condition levels.

### Key Business Questions

- Does vehicle condition significantly affect delivery time?
- Are delivery times significantly different across vehicle-condition groups?
- Should vehicle condition be considered when managing delivery operations?

In [7]:
vehicle_condition_groups = [
    group["Time_taken(min)"].dropna()
    for _, group in df.groupby("Vehicle_condition")
]

f_stat, p_value = stats.f_oneway(*vehicle_condition_groups)

print("F-statistic:", round(f_stat, 3))
print("P-value:", p_value)

F-statistic: 1323.323
P-value: 0.0


## 5. Delivery Partner Ratings vs Delivery Time

Test whether delivery times differ significantly between lower-rated and higher-rated delivery partners.

### Key Business Questions

- Do higher-rated delivery partners have significantly different delivery times?
- Is the difference in delivery time statistically significant?
- Is delivery partner rating associated with delivery performance?

In [8]:
median_rating = df["Delivery_person_Ratings"].median()

high_rating = df[
    df["Delivery_person_Ratings"] >= median_rating
]["Time_taken(min)"].dropna()

low_rating = df[
    df["Delivery_person_Ratings"] < median_rating
]["Time_taken(min)"].dropna()

t_stat, p_value = stats.ttest_ind(
    high_rating,
    low_rating,
    equal_var=False
)

print("Median rating:", median_rating)
print("High-rating group mean delivery time:", round(high_rating.mean(), 2))
print("Low-rating group mean delivery time:", round(low_rating.mean(), 2))
print("T-statistic:", round(t_stat, 3))
print("P-value:", p_value)

Median rating: 4.7
High-rating group mean delivery time: 24.33
Low-rating group mean delivery time: 29.0
T-statistic: -52.259
P-value: 0.0


## 6. Post-Hoc Analysis: Traffic Conditions

Identify which specific traffic conditions have statistically significant differences in delivery time after the ANOVA result.

### Key Business Questions

- Which traffic conditions differ significantly from each other?
- Is Jam traffic significantly slower than Low traffic?
- Which traffic conditions require the greatest operational attention?

In [4]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

traffic_data = df[
    ["Road_traffic_density", "Time_taken(min)"]
].dropna()

tukey_result = pairwise_tukeyhsd(
    endog=traffic_data["Time_taken(min)"],
    groups=traffic_data["Road_traffic_density"],
    alpha=0.05
)

print(tukey_result)

  Multiple Comparison of Means - Tukey HSD, FWER=0.05  
 group1  group2 meandiff p-adj   lower    upper  reject
-------------------------------------------------------
  High     Jam    3.9366    0.0    3.538  4.3352   True
  High     Low    -5.973    0.0  -6.3675 -5.5785   True
  High  Medium   -0.5404 0.0032  -0.9526 -0.1281   True
  High     NaN   -0.6976 0.3218  -1.7036  0.3084  False
   Jam     Low   -9.9096    0.0 -10.1788 -9.6405   True
   Jam  Medium    -4.477    0.0  -4.7716 -4.1824   True
   Jam     NaN   -4.6342    0.0   -5.598 -3.6704   True
   Low  Medium    5.4327    0.0   5.1437  5.7217   True
   Low     NaN    5.2755    0.0   4.3134  6.2375   True
Medium     NaN   -0.1572 0.9921  -1.1267  0.8123  False
-------------------------------------------------------


## 7. Distance vs Delivery Time

Test whether delivery distance has a statistically significant relationship with delivery time.

### Key Business Questions

- Is delivery distance significantly associated with delivery time?
- Does increasing distance lead to longer delivery times?
- Should distance be considered when estimating delivery times?

In [7]:
import numpy as np

lat1 = np.radians(df["Restaurant_latitude"])
lon1 = np.radians(df["Restaurant_longitude"])

lat2 = np.radians(df["Delivery_location_latitude"])
lon2 = np.radians(df["Delivery_location_longitude"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
)

df["Distance_km"] = 6371 * 2 * np.arcsin(np.sqrt(a))

print("Distance_km created successfully.")

Distance_km created successfully.


In [8]:
distance_data = df[
    ["Distance_km", "Time_taken(min)"]
].dropna()

r_stat, p_value = stats.pearsonr(
    distance_data["Distance_km"],
    distance_data["Time_taken(min)"]
)

print("Correlation coefficient:", round(r_stat, 3))
print("P-value:", p_value)

Correlation coefficient: -0.003
P-value: 0.5510260011828984


In [9]:
distance_filtered = df[
    (df["Distance_km"] <= 20) &
    (df["Distance_km"] > 0)
][["Distance_km", "Time_taken(min)"]].dropna()

r_stat, p_value = stats.pearsonr(
    distance_filtered["Distance_km"],
    distance_filtered["Time_taken(min)"]
)

print("Correlation coefficient:", round(r_stat, 3))
print("P-value:", p_value)

Correlation coefficient: 0.325
P-value: 0.0


## 7. Statistical Analysis Summary

The statistical tests confirm which operational factors have a significant relationship with delivery performance.

### Key Business Questions

- Which factors have a statistically significant impact on delivery time?
- Which factors show the strongest relationships?
- Which operational factors should be prioritized for improving delivery performance?

In [10]:
statistical_summary = pd.DataFrame({
    "Factor": [
        "Traffic Density",
        "Weather Condition",
        "Multiple Deliveries",
        "Vehicle Condition",
        "Delivery Partner Rating",
        "Distance (≤20 km)"
    ],
    "Test": [
        "One-way ANOVA",
        "One-way ANOVA",
        "One-way ANOVA",
        "One-way ANOVA",
        "Independent T-Test",
        "Pearson Correlation"
    ],
    "Statistic": [
        2550.296,
        608.220,
        2623.490,
        1323.323,
        -52.259,
        0.325
    ],
    "P_Value": [
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0
    ],
    "Significant": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]
})

statistical_summary

,Factor,Test,Statistic,P_Value,Significant
0,Traffic Density,One-way ANOVA,2550.296,0.0,Yes
1,Weather Condition,One-way ANOVA,608.220,0.0,Yes
2,Multiple Deliveries,One-way ANOVA,2623.490,0.0,Yes
3,Vehicle Condition,One-way ANOVA,1323.323,0.0,Yes
4,Delivery Partner Rating,Independent T-Test,-52.259,0.0,Yes
5,Distance (≤20 km),Pearson Correlation,0.325,0.0,Yes


### Statistical Interpretation

**1. Traffic Density**

The ANOVA test shows a statistically significant difference in delivery times across traffic conditions (F = 2550.296, p < 0.05). Delivery times increase substantially under heavier traffic, with Jam conditions having the highest average delivery time. This indicates that traffic is an important operational factor affecting delivery performance.

**Business insight:** Traffic conditions should be incorporated into ETA calculations, route planning, and delivery partner allocation.

---

**2. Weather Condition**

The ANOVA test shows a statistically significant difference in delivery times across weather conditions (F = 608.220, p < 0.05). Deliveries during Fog and Cloudy conditions take considerably longer than deliveries during Sunny conditions.

**Business insight:** Weather conditions should be considered when estimating ETAs and planning delivery capacity during unfavorable weather.

---

**3. Multiple Deliveries**

The ANOVA test shows a statistically significant difference in delivery times across multiple-delivery groups (F = 2623.490, p < 0.05). Delivery times increase substantially when delivery partners are assigned multiple orders, particularly for groups with 2 or more deliveries.

**Business insight:** Excessive multiple-order assignments should be controlled because they can substantially increase delivery delays.

---

**4. Vehicle Condition**

The ANOVA test shows a statistically significant difference in delivery times across vehicle-condition levels (F = 1323.323, p < 0.05). Vehicle condition also showed a negative correlation with delivery time, indicating that better vehicle condition is associated with faster deliveries.

**Business insight:** Regular vehicle maintenance and monitoring can support more reliable delivery performance.

---

**5. Delivery Partner Rating**

The independent t-test shows a statistically significant difference between higher-rated and lower-rated delivery partners (t = -52.259, p < 0.05). Higher-rated partners had an average delivery time of approximately 24.33 minutes compared with 29.00 minutes for lower-rated partners.

**Business insight:** Delivery partner performance should be monitored, and experienced high-performing partners can be prioritized for time-sensitive or high-risk deliveries.

---

**6. Delivery Distance**

For realistic delivery distances of 20 km or less, Pearson correlation shows a statistically significant positive relationship between distance and delivery time (r = 0.325, p < 0.05). This indicates that longer delivery distances are generally associated with longer delivery times.

The initial correlation using the raw distance data was approximately zero because extreme geographic coordinate anomalies created unrealistic distances. After restricting the analysis to realistic distances, the relationship became statistically significant.

**Business insight:** Delivery distance should be incorporated into ETA calculations, route planning, and order assignment.

### Overall Statistical Conclusion

The statistical analysis confirms that **traffic density, weather conditions, multiple deliveries, vehicle condition, delivery partner ratings, and realistic delivery distance are all statistically significant factors associated with delivery performance**.

Among these factors, traffic and multiple deliveries show particularly strong differences in delivery time, while delivery partner ratings and realistic delivery distance also demonstrate meaningful relationships with delivery performance.

These findings validate the patterns identified during exploratory analysis and provide statistical evidence that delivery delays are influenced by multiple operational factors rather than a single cause.

**Overall Business Insight:** The company should use a combination of traffic, weather, distance, vehicle condition, delivery partner performance, and current workload when estimating ETAs and assigning orders. A multi-factor approach would provide more reliable delivery-time predictions and help reduce late deliveries.